# **Data Preprocessing and Featurization(Example)**

## Install the required libraries

In [ ]:
# Install libraries to use matminer.
!pip install pyyaml -q
!pip install six -q
!pip install matminer[citrine] -q
!pip install citrination-client -q
!pip install --upgrade pandas==2.2.2 -q
!pip install pymatgen -q
!pip install --upgrade matplotlib==3.8.0 -q
!git clone https://github.com/CMDDclass/MS697-material.git

# Analyse the California house price dataset

In this notebook, you'll use the California house price dataset to perform data analysis and preprocessing.

## Step 0: Load the required libraries

In [ ]:
import numpy as np # a software library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.
import pandas as pd # a software library written for the Python programming language for data manipulation and analysis. In particular, it offers data structures and operations for manipulating numerical tables and time series
import matplotlib.pyplot as plt # a plotting library for the Python programming language
import seaborn as sns # Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics

# sklearn is a machine learning software library for the Python programming language. It features various classification, regression and clustering algorithms including support vector machines, random forests, gradient boosting,
# k-means and DBSCAN, and is designed to interoperate with the Python numerical and scientific libraries NumPy and SciPy.

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

# Ignore the warning message
import warnings
warnings.filterwarnings('ignore')

## Step 1: Data collecting

In [ ]:
# Load the California housing dataset
california = fetch_california_housing()

In [ ]:
# Check the description of dataset
print(california.DESCR)

In [ ]:
# Print the total row and column length of the dataset
california.data.shape

In [ ]:
# Print feature names
california.feature_names

## Step 2: Data pre-processing

In [ ]:
# Transforming the data set to Pandas' DataFrame formate (two-dimensional, size-mutable, potentially heterogeneous tabular data)
df = pd.DataFrame(california.data)

df.head()

In [ ]:
# Enter feature as the column name
df.columns = california.feature_names
df.head()

In [ ]:
# Put the target(price) in the dataset
df['Price'] = california.target

## Step 3: Data analysis

In [ ]:
# Generate descriptive statistics.
df.describe()

In [ ]:
# Check the distribution of dataset
df.hist(figsize=(12, 10), bins=100, edgecolor="black")
plt.subplots_adjust(hspace=0.7, wspace=0.4)

In [ ]:
# Pair plot between features
features = california.feature_names
features.append('Price')
grid = sns.pairplot(df[features])

From the plot, it is clear that MedInc (Median Income) and Price are correlated with each other.

In [ ]:
# Another way to plot this is using a correlation plot.
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
plt.title('Correlation Heatmap')
plt.show()

## Step 4: Feature selection

In [ ]:
## Function that returns feature names
def identify_columns(x_new, nrows=10):
    columns = x_data.columns
    xvalues = x_data.values
    dist = np.linalg.norm(xvalues[:nrows, :, None] - x_new[:nrows, None, :], axis=0)
    return columns[np.argmin(dist, axis=0)].values

In [ ]:
x_data = df[california.feature_names]
y_data = df['Price']

In [ ]:
sel = SelectKBest(f_regression, k=5)
x_new = sel.fit_transform(x_data, y_data)
print(f"Selected features {identify_columns(x_new)}")

## cf) **Standardization & Principal Component Analysis (PCA)**

Standardisation and PCA can give you another insight into your dataset.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Standardize the data
x_scaled = StandardScaler().fit_transform(x_data)

In [ ]:
# 2. Apply PCA
pca = PCA()
x_pca = pca.fit_transform(x_scaled)

In [ ]:
# 3. Explained Variance Ratio
explained_variance_ratio = pca.explained_variance_ratio_

In [ ]:
# 4. Visualize the explained variance ratio
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance Ratio by Principal Component')
plt.show()

In [ ]:
# 5. Determine the number of principal components to retain
# You can choose a threshold for the explained variance ratio (e.g., 95%)
cumulative_variance = np.cumsum(explained_variance_ratio)
n_components_to_retain = np.argmax(cumulative_variance >= 0.95) + 1

In [ ]:
# 6. Analyze the loadings of the principal components
loadings = pca.components_
feature_names = x_data.columns

# Create a DataFrame for the loadings
loadings_df = pd.DataFrame(loadings, columns=feature_names)

# Print the loadings for the retained principal components
print(f"Loadings for the first {n_components_to_retain} principal components:")
print(loadings_df.iloc[:n_components_to_retain])

In [ ]:
# 7. Interpret the results
# The loadings show the correlation between the original features and the principal components.
# Features with high absolute loadings contribute more to the variance explained by that principal component.
# By examining the loadings, you can identify which features have the most significant impact on the principal components, and thus on the y_data (Price) in this case.

print(f"\nBased on the PCA analysis, the following features seem to have the most significant impact on the 'Price':")
for i in range(n_components_to_retain):
  top_features = loadings_df.iloc[i].abs().nlargest(3).index.tolist()
  print(f"Principal Component {i+1}: {', '.join(top_features)}")

While "Median Income" stands out as a primary factor on its own, from the perspective of the principal component analysis, it becomes apparent that a combination of other factors—"AveRooms, AveBedrms, Latitude"—also plays a significant role.